[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No-Country-simulation/G10-LATAM-EQUIPO-40/blob/main/notebook/agent_prototype.ipynb)

In [3]:
#!pip install -U langchain
#!pip install -U langchain-google-genai langchain-cohere langchain-community langchain-core

In [ ]:
# Importar en Colab

from google.colab import userdata
COHERE_API_KEY = userdata.get("COHERE_API_KEY")

In [ ]:
# De forma Local
# Importar la API de Cohere: Crear un .env con una variable COHERE_API_KEY = "TU-API"
# Poner el .env al .gitignore 

import os
from dotenv import load_dotenv

load_dotenv()
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

In [5]:
# Crear el agente

from langchain_cohere import ChatCohere
from langchain_core.messages import AIMessage, HumanMessage

import base64, mimetypes
from langchain_cohere import ChatCohere
from langchain_core.messages import HumanMessage

def to_data_url(path):
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    with open(path, "rb") as f:
        return f"data:{mime};base64,{base64.b64encode(f.read()).decode()}"

llm = ChatCohere(
    cohere_api_key=COHERE_API_KEY,
    model="command-a-vision-07-2025",
)

In [6]:
# Esquema de salida, primera prueba

schema = {
    "title": "DocumentoMedico",
    "type": "object",
    "properties": {
        "paciente": {"type": "string"},
        "fecha": {"type": "string"},
        "medico": {"type": "string"},
        "diagnostico": {"type": "string"},
        "medicamentos": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["paciente", "fecha"],
}

In [ ]:
# Primer prompt
# Poner el archivo en la carpeta raiz con el nombre --documento.jpg--

msg = HumanMessage(content=[
    {"type": "text", "text": "Extrae los campos del documento. Si un campo no aparece, déjalo vacío. No inventes datos."},
    {"type": "image_url", "image_url": {"url": to_data_url("../documento.jpg")}},
])

In [10]:
# Primera invocación

res = llm.invoke([msg], response_format={"type": "json_object", "schema": schema})
print(res.content)

{
  "title": "DocumentoMedico",
  "type": "object",
  "properties": {
    "paciente": "JUAN PABLO HERRERA LÓPEZ",
    "fecha": "10/09/2026",
    "medico": "Dra. Laura Patricia Mejía",
    "diagnostico": "",
    "medicamentos": []
  },
  "required": ["paciente", "fecha"]
}
